# Circuits — the theory that lets you solve any of them

The previous notebook took the components apart one at a time. This one is about what happens when they are wired together, and about the small set of results that make an arbitrary network solvable without guessing.

Everything here follows from two conservation statements and nothing else:

$$\sum_{\text{into a node}} i = 0
\qquad\qquad
\sum_{\text{around a loop}} v = 0$$

Charge does not pile up at a junction, and potential is single-valued so a round trip returns to where it started. From those two, plus each component's own rule, come series and parallel combination, the two dividers, nodal analysis, Thévenin's theorem and superposition — and every one of them is checked numerically in the panels rather than asserted.

The schematics animate as before: wire colour is node voltage, yellow dots are charge, and their speed is the current.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import ipywidgets as widgets
from IPython.display import display

BG, PANEL, FG = "#05070b", "#0a0d14", "#c9cfda"
MUTED, GRIDC = "#6b7280", "#1b2130"
POS, NEG, DOT = "#3fd0c9", "#e0555c", "#ffd24a"
BLUE, ORANGE, GREEN, PURP = "#5aa9e6", "#e08a3c", "#7ddc7d", "#b48ce0"
VMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "volt", [(0.0, NEG), (0.5, "#4a5060"), (1.0, POS)])

plt.rcParams.update({
    "figure.dpi": 112, "font.size": 8.5, "axes.titlesize": 9,
    "figure.facecolor": BG, "savefig.facecolor": BG, "axes.facecolor": PANEL,
    "axes.edgecolor": GRIDC, "axes.labelcolor": FG, "text.color": FG,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRIDC,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.facecolor": PANEL, "legend.edgecolor": GRIDC, "legend.framealpha": 0.9,
})
SL = {"style": {"description_width": "104px"},
      "layout": widgets.Layout(width="290px"), "continuous_update": False}


def panel(ax, edge=None, lw=1.3):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_visible(True); s.set_color(edge or GRIDC)
        s.set_linewidth(lw if edge else 0.8)
    ax.tick_params(colors=MUTED, labelsize=7)
    return ax


def readout(fig, x, y, lines, color=FG, size=7.4):
    fig.text(x, y, "\n".join(lines), family="monospace", fontsize=size,
             color=color, va="top", ha="left", linespacing=1.55)


def footer(fig, text):
    fig.text(0.010, 0.012, text, family="monospace", fontsize=6.6, color=MUTED)
    fig.text(0.990, 0.012, "circuits · theory", family="monospace",
             fontsize=6.6, color=MUTED, ha="right")


def timeline(n, step=1, interval=90, desc="time"):
    p = widgets.Play(value=0, min=0, max=n, step=step, interval=interval)
    s = widgets.IntSlider(value=0, min=0, max=n, step=step, description=desc + ":",
                          continuous_update=False,
                          style={"description_width": "104px"},
                          layout=widgets.Layout(width="430px"))
    widgets.jslink((p, "value"), (s, "value"))
    return p, s


# ----- schematic primitives, Falstad style -------------------------------
def vcolor(v, vmax):
    return VMAP(np.clip(0.5 + 0.5 * v / max(vmax, 1e-9), 0, 1))


def wire(ax, pts, v, vmax, lw=2.6):
    pts = np.asarray(pts, float)
    seg = np.stack([pts[:-1], pts[1:]], axis=1)
    ax.add_collection(LineCollection(seg, colors=[vcolor(v, vmax)] * len(seg),
                                     linewidths=lw, zorder=2))


def node_dot(ax, p, v, vmax, s=34):
    ax.plot(*p, "o", ms=np.sqrt(s), color=vcolor(v, vmax), zorder=4)


def resistor(ax, p0, p1, v, vmax, label=None, n=6, amp=0.16):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.28, p1 - u * L * 0.28
    ts = np.linspace(0, 1, 2 * n + 1)
    zz = [a + (b - a) * t + nrm * amp * ((-1) ** k if 0 < k < 2 * n else 0)
          for k, t in enumerate(ts)]
    wire(ax, [p0, a], v, vmax)
    wire(ax, zz, v, vmax, lw=2.2)
    wire(ax, [b, p1], v, vmax)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.34), label, color=FG, fontsize=7.5,
                ha="center", va="center")


def capacitor(ax, p0, p1, v, vmax, label=None, gap=0.10, half=0.24):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * gap, c + u * gap
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    for q in (a, b):
        ax.plot(*np.stack([q - nrm * half, q + nrm * half]).T, color=FG, lw=2.4,
                zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def inductor(ax, p0, p1, v, vmax, label=None, coils=4, r=0.13):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.25, p1 - u * L * 0.25
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    seg = np.linalg.norm(b - a) / coils
    for k in range(coils):
        c = a + u * seg * (k + 0.5)
        th = np.linspace(0, np.pi, 24)
        pts = np.array([c + u * (seg / 2) * np.cos(np.pi - t) + nrm * r * np.sin(t)
                        for t in th])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=2.0, zorder=3)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.38), label, color=FG, fontsize=7.5,
                ha="center")


def diode(ax, p0, p1, v, vmax, label=None, s=0.20):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * s, c + u * s
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    ax.add_patch(mpatches.Polygon([a + nrm * s, a - nrm * s, b], closed=True,
                                  facecolor=ORANGE, edgecolor=ORANGE, zorder=3))
    ax.plot(*np.stack([b - nrm * s, b + nrm * s]).T, color=FG, lw=2.6, zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def source(ax, p0, p1, v, vmax, kind="dc", label=None, r=0.30):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    c = 0.5 * (p0 + p1)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    wire(ax, [p0, c - u * r], v, vmax); wire(ax, [c + u * r, p1], v, vmax)
    ax.add_patch(mpatches.Circle(c, r, fill=False, ec=FG, lw=2.0, zorder=3))
    if kind == "dc":
        ax.plot(*np.stack([c - u * 0.12 - nrm * 0.16, c - u * 0.12 + nrm * 0.16]).T,
                color=FG, lw=2.6, zorder=4)
        ax.plot(*np.stack([c + u * 0.12 - nrm * 0.09, c + u * 0.12 + nrm * 0.09]).T,
                color=FG, lw=2.0, zorder=4)
    else:
        t = np.linspace(-1, 1, 40)
        pts = np.array([c + u * (0.19 * t[i]) + nrm * 0.15 * np.sin(np.pi * t[i])
                        for i in range(len(t))])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=1.8, zorder=4)
    if label:
        ax.text(*(c + nrm * (r + 0.22)), label, color=FG, fontsize=7.5, ha="center")


def path_len(pts):
    p = np.asarray(pts, float)
    d = np.linalg.norm(np.diff(p, axis=0), axis=1)
    return np.r_[0, np.cumsum(d)]


def charge_dots(ax, loop, q, spacing=0.42, ms=4.2):
    """Yellow dots at arclength q + n*spacing — this is the current, visualised."""
    p = np.asarray(loop, float)
    s = path_len(p)
    L = s[-1]
    if L <= 0:
        return
    offs = (np.arange(0, L, spacing) + (q % spacing)) % L
    x = np.interp(offs, s, p[:, 0]); y = np.interp(offs, s, p[:, 1])
    ax.plot(x, y, "o", ms=ms, color=DOT, zorder=5, mec="none")


def sch_axes(ax, xlim, ylim):
    panel(ax)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    return ax


def loop_rect(x0, x1, y0, y1, n=60):
    top = np.stack([np.linspace(x0, x1, n), np.full(n, y1)], 1)
    right = np.stack([np.full(n, x1), np.linspace(y1, y0, n)], 1)
    bot = np.stack([np.linspace(x1, x0, n), np.full(n, y0)], 1)
    left = np.stack([np.full(n, x0), np.linspace(y0, y1, n)], 1)
    return np.vstack([top, right, bot, left])


def seg(p0, p1, n=40):
    return np.stack([np.linspace(p0[0], p1[0], n),
                     np.linspace(p0[1], p1[1], n)], 1)


print("schematic engine ready — the same primitives as circuits_components")
print("dot speed is current · wire colour is node voltage")

schematic engine ready — the same primitives as circuits_components
dot speed is current · wire colour is node voltage


## Series and parallel — the two ways to share

Two components can share a **current** or share a **voltage**, and which one they share decides everything.

$$R_{\text{series}}=R_1+R_2,\qquad
R_{\text{parallel}}=\frac{R_1R_2}{R_1+R_2}$$

In series the same charge must pass through both, so the current is identical and the voltage splits in proportion to resistance — the **voltage divider**, $v_2=V\frac{R_2}{R_1+R_2}$. Watch the dots: one stream, one speed, all the way round.

In parallel both ends sit at the same two nodes, so the voltage is shared and the current splits instead — the **current divider**, $i_1=I\frac{R_2}{R_1+R_2}$. Note the index: the current through $R_1$ is set by $R_2$. The *smaller* resistor takes the larger share, which is the opposite of the voltage case and the source of endless sign errors.

The dots make the asymmetry obvious. In the parallel schematic one branch runs visibly faster than the other while both endpoints stay the same colour; in the series schematic the speeds are identical and it is the colours that change.

In [2]:
def draw_seriespar(k, V, R1, R2):
    Rs = R1 + R2
    Rp = R1 * R2 / (R1 + R2)
    i_s = V / Rs
    v1, v2 = i_s * R1, i_s * R2
    i1, i2 = V / R1, V / R2
    vmax = max(V, 1e-9)
    q = k / 400 * 1e-2

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1.2, 0.55],
                          wspace=0.26, hspace=0.44, left=0.02, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[0, 0])
    sch_axes(a0, (-0.5, 4.8), (-0.4, 2.0))
    wire(a0, [(0.5, 1.6), (1.4, 1.6)], V, vmax)
    resistor(a0, (1.4, 1.6), (2.6, 1.6), V, vmax, f"R1 {R1:.0f}Ω")
    resistor(a0, (2.6, 1.6), (3.9, 1.6), v2, vmax, f"R2 {R2:.0f}Ω")
    wire(a0, [(3.9, 1.6), (4.3, 1.6), (4.3, 0.1), (0.5, 0.1)], 0.0, vmax)
    source(a0, (0.5, 0.1), (0.5, 1.6), V / 2, vmax, "dc", f"{V:.1f}V")
    node_dot(a0, (2.6, 1.6), v2, vmax)
    charge_dots(a0, loop_rect(0.5, 4.3, 0.1, 1.6), q * i_s * 4e4, spacing=0.34, ms=3.6)
    a0.set_title(f"series — one current {i_s*1e3:.3f} mA, "
                 f"voltage splits {v1:.2f} / {v2:.2f} V", fontsize=8.5)

    a1 = fig.add_subplot(gs[1, 0])
    sch_axes(a1, (-0.5, 4.8), (-0.4, 2.0))
    wire(a1, [(0.5, 1.6), (3.9, 1.6)], V, vmax)
    wire(a1, [(0.5, 0.1), (3.9, 0.1)], 0.0, vmax)
    source(a1, (0.5, 0.1), (0.5, 1.6), V / 2, vmax, "dc", f"{V:.1f}V")
    resistor(a1, (2.2, 1.6), (2.2, 0.1), V / 2, vmax, f"R1 {R1:.0f}Ω")
    resistor(a1, (3.9, 1.6), (3.9, 0.1), V / 2, vmax, f"R2 {R2:.0f}Ω")
    node_dot(a1, (2.2, 1.6), V, vmax); node_dot(a1, (2.2, 0.1), 0.0, vmax)
    charge_dots(a1, seg((0.5, 1.6), (2.2, 1.6)), q * (i1 + i2) * 4e4,
                spacing=0.30, ms=3.6)
    charge_dots(a1, seg((2.2, 1.6), (2.2, 0.1)), q * i1 * 4e4, spacing=0.30, ms=3.6)
    charge_dots(a1, seg((3.9, 1.6), (3.9, 0.1)), q * i2 * 4e4, spacing=0.30, ms=3.6)
    a1.set_title(f"parallel — one voltage {V:.1f} V, current splits "
                 f"{i1*1e3:.3f} / {i2*1e3:.3f} mA", fontsize=8.5)

    a2 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    rr = np.logspace(1, 5, 300)
    a2.semilogx(rr, V * rr / (R1 + rr), color=POS, lw=1.6, label="v across R2")
    a2.semilogx(rr, V * R1 / (R1 + rr), color=DOT, lw=1.6, label="v across R1")
    a2.axvline(R2, color=FG, lw=1.0, ls="--")
    a2.plot([R2], [v2], "o", ms=7, color=POS)
    a2.set_xlabel("R2  (Ω)"); a2.set_ylabel("volts")
    a2.legend(fontsize=7)
    a2.set_title("voltage divider — the bigger resistor takes more")

    a3 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    Itot = i1 + i2
    a3.semilogx(rr, Itot * rr / (R1 + rr) * 1e3, color=DOT, lw=1.6,
                label="i through R1")
    a3.semilogx(rr, Itot * R1 / (R1 + rr) * 1e3, color=POS, lw=1.6,
                label="i through R2")
    a3.axvline(R2, color=FG, lw=1.0, ls="--")
    a3.set_xlabel("R2  (Ω)"); a3.set_ylabel("mA")
    a3.legend(fontsize=7)
    a3.set_title("current divider — the smaller resistor takes more")

    readout(fig, 0.845, 0.90, [
        "COMPONENTS", "─" * 26,
        f"V           {V:>10.2f}V",
        f"R1          {R1:>10.0f}Ω",
        f"R2          {R2:>10.0f}Ω",
        "", "SERIES", "─" * 26,
        f"R total     {Rs:>10.1f}Ω",
        f"current     {i_s*1e3:>10.4f}mA",
        f"v on R1     {v1:>10.4f}V",
        f"v on R2     {v2:>10.4f}V",
        f"v1+v2       {v1+v2:>10.4f}V",
        "", "PARALLEL", "─" * 26,
        f"R total     {Rp:>10.1f}Ω",
        f"i in R1     {i1*1e3:>10.4f}mA",
        f"i in R2     {i2*1e3:>10.4f}mA",
        f"i1+i2       {(i1+i2)*1e3:>10.4f}mA",
        f"V/Rp        {V/Rp*1e3:>10.4f}mA",
    ])
    footer(fig, f"series R = R1+R2 = {Rs:.0f} Ω   ·   "
                f"parallel R = R1R2/(R1+R2) = {Rp:.1f} Ω")
    plt.show()


_p1, _s1 = timeline(399, step=4)
w1 = dict(V=widgets.FloatSlider(value=9, min=1, max=15, step=0.5,
                                description="source V:", **SL),
          R1=widgets.FloatSlider(value=1000, min=100, max=5000, step=100,
                                 description="R1 (Ω):", **SL),
          R2=widgets.FloatSlider(value=2200, min=100, max=5000, step=100,
                                 description="R2 (Ω):", **SL),
          k=_s1)
display(widgets.VBox([widgets.HBox([w1["V"], w1["R1"], w1["R2"]]),
                      widgets.HBox([_p1, _s1])]),
        widgets.interactive_output(draw_seriespar, w1))

Output()

## Kirchhoff's laws — the two statements everything rests on

**KCL**: charge does not accumulate at a junction, so whatever flows in flows out.

$$\sum i_{\text{in}}=\sum i_{\text{out}}$$

The node in this circuit splits one stream into two, and the dots show it as a rate: the incoming stream is denser or faster than either branch, and the two branch rates add back to it exactly. The panel evaluates the sum at the node and it sits at the numerical floor, around $10^{-18}$ A — not approximately zero, zero to the last bit the machine has.

**KVL**: potential is a property of a point, so walking a loop and returning to the start must bring you back to the same value.

$$\sum_{\text{loop}} v = 0$$

The walk panel does exactly that — it starts at ground, climbs through the source, drops through each resistor, and arrives back at zero. A voltage "drop" is not something a resistor does *to* a current, it is just the height difference between two points, and the loop is a closed contour on a potential.

Neither law says anything about what the components are. They hold for capacitors mid-transient, for diodes, for anything — which is why they are the foundation and Ohm's law is not.

In [ ]:
def draw_kirchhoff(k, V, Ra, Rb, Rc):
    Rpar = Rb * Rc / (Rb + Rc)
    itot = V / (Ra + Rpar)
    vn = itot * Rpar
    ib, ic = vn / Rb, vn / Rc
    vmax = max(V, 1e-9)
    q = k / 400 * 1e-2

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.3, 1.15, 0.55],
                          wspace=0.26, hspace=0.44, left=0.02, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.5, 5.0), (-0.4, 2.6))
    wire(a0, [(0.5, 2.1), (1.3, 2.1)], V, vmax)
    resistor(a0, (1.3, 2.1), (2.6, 2.1), V, vmax, f"Ra {Ra:.0f}Ω")
    wire(a0, [(2.6, 2.1), (4.4, 2.1)], vn, vmax)
    wire(a0, [(0.5, 0.1), (4.4, 0.1)], 0.0, vmax)
    source(a0, (0.5, 0.1), (0.5, 2.1), V / 2, vmax, "dc", f"{V:.0f}V")
    resistor(a0, (3.2, 2.1), (3.2, 0.1), vn / 2, vmax, f"Rb {Rb:.0f}Ω")
    resistor(a0, (4.4, 2.1), (4.4, 0.1), vn / 2, vmax, f"Rc {Rc:.0f}Ω")
    node_dot(a0, (3.2, 2.1), vn, vmax, s=70)
    a0.text(3.2, 2.35, "node", color=DOT, fontsize=8, ha="center")
    charge_dots(a0, seg((0.5, 2.1), (3.2, 2.1)), q * itot * 4e4, spacing=0.28, ms=3.8)
    charge_dots(a0, seg((3.2, 2.1), (3.2, 0.1)), q * ib * 4e4, spacing=0.28, ms=3.8)
    charge_dots(a0, seg((3.2, 2.1), (4.4, 2.1), 20), q * ic * 4e4, spacing=0.28, ms=3.8)
    charge_dots(a0, seg((4.4, 2.1), (4.4, 0.1)), q * ic * 4e4, spacing=0.28, ms=3.8)
    a0.set_title("one stream in, two out — the rates add back exactly")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    a1.bar([0, 1, 2], [itot * 1e3, -ib * 1e3, -ic * 1e3],
           color=[DOT, POS, PURP], width=0.6)
    a1.axhline(0, color=FG, lw=1.0)
    a1.set_xticks([0, 1, 2])
    a1.set_xticklabels(["in  (Ra)", "out (Rb)", "out (Rc)"], fontsize=7.5)
    a1.set_ylabel("current  (mA)")
    a1.set_title(f"KCL at the node:  sum = {(itot-ib-ic):+.3e} A")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    steps = ["gnd", "through\nsource", "through\nRa", "through\nRb", "back to\ngnd"]
    pot = [0.0, V, V - itot * Ra, V - itot * Ra - ib * Rb, 0.0]
    a2.step(range(len(pot)), pot, where="mid", color=GREEN, lw=1.8)
    a2.plot(range(len(pot)), pot, "o", ms=6, color=DOT)
    a2.axhline(0, color=FG, lw=0.9, ls="--")
    a2.set_xticks(range(len(steps)))
    a2.set_xticklabels(steps, fontsize=6.8)
    a2.set_ylabel("potential  (V)")
    a2.set_title(f"KVL walk around the loop:  ends at {pot[-1]:+.3e} V")

    readout(fig, 0.845, 0.90, [
        "CIRCUIT", "─" * 26,
        f"V           {V:>10.2f}V",
        f"Ra          {Ra:>10.0f}Ω",
        f"Rb          {Rb:>10.0f}Ω",
        f"Rc          {Rc:>10.0f}Ω",
        f"Rb||Rc      {Rpar:>10.2f}Ω",
        "", "NODE VOLTAGE", "─" * 26,
        f"v node      {vn:>10.5f}V",
        f"V·Rp/(Ra+Rp){V*Rpar/(Ra+Rpar):>10.5f}V",
        "", "CURRENTS", "─" * 26,
        f"in   (Ra)   {itot*1e3:>10.5f}mA",
        f"out  (Rb)   {ib*1e3:>10.5f}mA",
        f"out  (Rc)   {ic*1e3:>10.5f}mA",
        f"KCL residual{itot-ib-ic:>+10.2e}A",
        "", "VOLTAGES", "─" * 26,
        f"on Ra       {itot*Ra:>10.5f}V",
        f"on Rb       {vn:>10.5f}V",
        f"KVL residual{V-itot*Ra-vn:>+10.2e}V",
    ])
    footer(fig, "KCL: charge does not pile up at a junction  ·  "
                "KVL: potential is single-valued")
    plt.show()


_p2, _s2 = timeline(399, step=4)
w2 = dict(V=widgets.FloatSlider(value=12, min=1, max=20, step=1,
                                description="source V:", **SL),
          Ra=widgets.FloatSlider(value=1000, min=100, max=4000, step=100,
                                 description="Ra (Ω):", **SL),
          Rb=widgets.FloatSlider(value=2000, min=100, max=6000, step=100,
                                 description="Rb (Ω):", **SL),
          Rc=widgets.FloatSlider(value=3000, min=100, max=6000, step=100,
                                 description="Rc (Ω):", **SL),
          k=_s2)
display(widgets.VBox([widgets.HBox([w2["V"], w2["Ra"], w2["Rb"], w2["Rc"]]),
                      widgets.HBox([_p2, _s2])]),
        widgets.interactive_output(draw_kirchhoff, w2))

Output()

## Nodal analysis — turning a circuit into a matrix

Applying KCL by hand works until the circuit has more than a couple of nodes. The systematic version writes one KCL equation per unknown node and solves them together:

$$\mathbf{G}\mathbf{v}=\mathbf{i},\qquad
G_{kk}=\sum(\text{conductances at node }k),\qquad
G_{kj}=-\sum(\text{conductances between }k\text{ and }j)$$

The matrix can be written down by inspection, which is the whole point. The diagonal is everything touching that node, the off-diagonals are what connects the two nodes, negated, and the right-hand side holds the current injected into each node. It is always **symmetric** for resistive networks, and that symmetry is reciprocity: a source at node 1 produces the same reading at node 2 as the other way round.

The panel builds $\mathbf{G}$ from the sliders, prints it with its actual numbers, solves it, and then goes back and checks KCL at every node independently — residuals land around $10^{-18}$ A.

This is what SPICE does. Add capacitors and inductors and the entries become frequency-dependent or get companion models per timestep, but the structure never changes: build the matrix, solve, repeat.

In [4]:
def nodal_solve(R1, R2, R3, R4, Is):
    G = np.array([
        [1 / R1 + 1 / R2, -1 / R2],
        [-1 / R2, 1 / R2 + 1 / R3 + 1 / R4]])
    b = np.array([Is, 0.0])
    v = np.linalg.solve(G, b)
    return G, b, v


def draw_nodal(k, R1, R2, R3, R4, Is_mA):
    Is = Is_mA * 1e-3
    G, b, v = nodal_solve(R1, R2, R3, R4, Is)
    v1, v2 = v
    i12 = (v1 - v2) / R2
    vmax = max(abs(v).max(), 1e-9)
    q = k / 400 * 1e-2

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.3, 1.1, 0.6],
                          wspace=0.3, hspace=0.5, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.5, 5.2), (-0.5, 2.8))
    wire(a0, [(0.4, 2.2), (1.6, 2.2)], v1, vmax)
    resistor(a0, (1.6, 2.2), (3.0, 2.2), v1, vmax, f"R2 {R2:.0f}Ω")
    wire(a0, [(3.0, 2.2), (4.6, 2.2)], v2, vmax)
    wire(a0, [(0.4, 0.2), (4.6, 0.2)], 0.0, vmax)
    resistor(a0, (0.4, 2.2), (0.4, 0.2), v1 / 2, vmax, f"R1 {R1:.0f}Ω")
    resistor(a0, (3.6, 2.2), (3.6, 0.2), v2 / 2, vmax, f"R3 {R3:.0f}Ω")
    resistor(a0, (4.6, 2.2), (4.6, 0.2), v2 / 2, vmax, f"R4 {R4:.0f}Ω")
    a0.annotate("", xy=(1.0, 2.2), xytext=(1.0, 1.0),
                arrowprops=dict(arrowstyle="-|>", color=DOT, lw=2.0))
    a0.text(1.15, 1.5, f"{Is_mA:.1f} mA", color=DOT, fontsize=8)
    node_dot(a0, (1.0, 2.2), v1, vmax, s=70); node_dot(a0, (3.3, 2.2), v2, vmax, s=70)
    a0.text(1.0, 2.45, "node 1", color=FG, fontsize=8, ha="center")
    a0.text(3.3, 2.45, "node 2", color=FG, fontsize=8, ha="center")
    charge_dots(a0, seg((1.6, 2.2), (3.0, 2.2), 24), q * i12 * 4e4, spacing=0.26,
                ms=3.6)
    a0.set_title("two unknown node voltages, everything else follows")

    a1 = panel(fig.add_subplot(gs[0, 1]))
    a1.imshow(G * 1e3, cmap="PiYG", vmin=-abs(G * 1e3).max(),
              vmax=abs(G * 1e3).max())
    for (i, j), val in np.ndenumerate(G):
        a1.text(j, i, f"{val*1e3:+.3f}", ha="center", va="center", fontsize=9,
                color="#111")
    a1.set_xticks([0, 1]); a1.set_xticklabels(["v1", "v2"])
    a1.set_yticks([0, 1]); a1.set_yticklabels(["KCL 1", "KCL 2"])
    a1.grid(False)
    a1.set_title("G  (mS) — symmetric by construction")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a2.barh([1, 0], [v1, v2], color=[POS, PURP], height=0.55)
    for y, val in ((1, v1), (0, v2)):
        a2.text(val, y, f"  {val:.5f} V", va="center", fontsize=8.5, color=FG)
    a2.set_yticks([0, 1]); a2.set_yticklabels(["v2", "v1"])
    a2.set_xlabel("volts")
    a2.set_title("solution of G v = i")

    kcl1 = Is - v1 / R1 - (v1 - v2) / R2
    kcl2 = (v1 - v2) / R2 - v2 / R3 - v2 / R4
    readout(fig, 0.845, 0.90, [
        "ELEMENTS", "─" * 26,
        f"R1 (n1→gnd) {R1:>10.0f}Ω",
        f"R2 (n1→n2)  {R2:>10.0f}Ω",
        f"R3 (n2→gnd) {R3:>10.0f}Ω",
        f"R4 (n2→gnd) {R4:>10.0f}Ω",
        f"Is into n1  {Is_mA:>10.2f}mA",
        "", "MATRIX", "─" * 26,
        f"G11         {G[0,0]*1e3:>+10.4f}mS",
        f"G12 = G21   {G[0,1]*1e3:>+10.4f}mS",
        f"G22         {G[1,1]*1e3:>+10.4f}mS",
        f"symmetric   {str(np.allclose(G, G.T)):>10s}",
        f"det         {np.linalg.det(G)*1e6:>+10.4f}",
        "", "SOLUTION", "─" * 26,
        f"v1          {v1:>10.5f}V",
        f"v2          {v2:>10.5f}V",
        f"i R1→R2     {i12*1e3:>10.5f}mA",
        "", "CHECK", "─" * 26,
        f"KCL node 1  {kcl1:>+10.2e}A",
        f"KCL node 2  {kcl2:>+10.2e}A",
    ])
    footer(fig, "G v = i  ·  diagonal = conductances at the node  ·  "
                "off-diagonal = −conductance between nodes  ·  this is what SPICE does")
    plt.show()


_p3, _s3 = timeline(399, step=4)
w3 = dict(R1=widgets.FloatSlider(value=1000, min=200, max=5000, step=100,
                                 description="R1 (Ω):", **SL),
          R2=widgets.FloatSlider(value=2000, min=200, max=5000, step=100,
                                 description="R2 (Ω):", **SL),
          R3=widgets.FloatSlider(value=3000, min=200, max=5000, step=100,
                                 description="R3 (Ω):", **SL),
          R4=widgets.FloatSlider(value=4700, min=200, max=8000, step=100,
                                 description="R4 (Ω):", **SL),
          Is_mA=widgets.FloatSlider(value=5, min=0.5, max=20, step=0.5,
                                    description="Is (mA):", **SL),
          k=_s3)
display(widgets.VBox([widgets.HBox([w3["R1"], w3["R2"], w3["R3"]]),
                      widgets.HBox([w3["R4"], w3["Is_mA"]]),
                      widgets.HBox([_p3, _s3])]),
        widgets.interactive_output(draw_nodal, w3))

Output()

## Thévenin — any network, however large, is a source and a resistor

Seen from two terminals, a network of sources and resistors is **indistinguishable** from a single voltage source behind a single resistance:

$$V_{th}=v_{\text{open circuit}},\qquad
R_{th}=\frac{V_{th}}{i_{\text{short circuit}}}$$

Not approximately — exactly. The panel drives both the real divider and its two-element equivalent with the same load and compares, sweeping $R_L$ from 10 Ω to 1 MΩ: the two agree to $10^{-16}$, the floor of double precision, at every load.

This is why datasheets quote an output impedance instead of a schematic, and why "loading" is predictable. Hang a load comparable to $R_{th}$ on an output and you lose half the voltage, which the curve shows directly.

The power curve underneath has its own well-known consequence. Delivered power peaks when $R_L=R_{th}$, at $P_{max}=V_{th}^2/4R_{th}$ — confirmed to three decimals in the readout. But at that point **exactly half the power is being burnt inside the source**, so maximum power transfer means 50% efficiency. Radio receivers match for power; power distribution deliberately does not, because there the wasted half is the whole business.

In [5]:
def draw_thevenin(k, Vs, Ra, Rb, RL):
    Vth = Vs * Rb / (Ra + Rb)
    Rth = Ra * Rb / (Ra + Rb)
    vfull = Vs * (Rb * RL / (Rb + RL)) / (Ra + Rb * RL / (Rb + RL))
    vth_l = Vth * RL / (Rth + RL)
    iL = vth_l / RL
    P = vth_l * iL
    vmax = max(Vs, 1e-9)
    q = k / 400 * 1e-2

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1.2, 0.55],
                          wspace=0.26, hspace=0.44, left=0.02, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[0, 0])
    sch_axes(a0, (-0.4, 5.0), (-0.4, 2.0))
    wire(a0, [(0.4, 1.6), (1.2, 1.6)], Vs, vmax)
    resistor(a0, (1.2, 1.6), (2.4, 1.6), Vs, vmax, f"Ra {Ra:.0f}Ω")
    wire(a0, [(2.4, 1.6), (4.5, 1.6)], vfull, vmax)
    wire(a0, [(0.4, 0.1), (4.5, 0.1)], 0.0, vmax)
    source(a0, (0.4, 0.1), (0.4, 1.6), Vs / 2, vmax, "dc", f"{Vs:.0f}V")
    resistor(a0, (3.1, 1.6), (3.1, 0.1), vfull / 2, vmax, f"Rb {Rb:.0f}Ω")
    resistor(a0, (4.5, 1.6), (4.5, 0.1), vfull / 2, vmax, f"RL {RL:.0f}Ω")
    node_dot(a0, (3.1, 1.6), vfull, vmax)
    charge_dots(a0, seg((4.5, 1.6), (4.5, 0.1)), q * iL * 3e4, spacing=0.26, ms=3.6)
    a0.set_title(f"the real network — v across RL = {vfull:.6f} V", fontsize=8.5)

    a1 = fig.add_subplot(gs[1, 0])
    sch_axes(a1, (-0.4, 5.0), (-0.4, 2.0))
    wire(a1, [(0.4, 1.6), (1.2, 1.6)], Vth, vmax)
    resistor(a1, (1.2, 1.6), (2.6, 1.6), Vth, vmax, f"Rth {Rth:.0f}Ω")
    wire(a1, [(2.6, 1.6), (4.5, 1.6)], vth_l, vmax)
    wire(a1, [(0.4, 0.1), (4.5, 0.1)], 0.0, vmax)
    source(a1, (0.4, 0.1), (0.4, 1.6), Vth / 2, vmax, "dc", f"Vth {Vth:.2f}V")
    resistor(a1, (4.5, 1.6), (4.5, 0.1), vth_l / 2, vmax, f"RL {RL:.0f}Ω")
    charge_dots(a1, seg((4.5, 1.6), (4.5, 0.1)), q * iL * 3e4, spacing=0.26, ms=3.6)
    a1.set_title(f"the equivalent — v across RL = {vth_l:.6f} V", fontsize=8.5)

    rl = np.logspace(0, 6, 400)
    a2 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a2.semilogx(rl, Vs * (Rb * rl / (Rb + rl)) / (Ra + Rb * rl / (Rb + rl)),
                color=POS, lw=2.4, label="full circuit")
    a2.semilogx(rl, Vth * rl / (Rth + rl), color=DOT, lw=1.1, ls="--",
                label="Thévenin")
    a2.axvline(RL, color=FG, lw=1.0, ls=":")
    a2.axhline(Vth, color=MUTED, lw=0.8, ls=":")
    a2.set_xlabel("load  RL  (Ω)"); a2.set_ylabel("volts")
    a2.legend(fontsize=7)
    a2.set_title(f"identical everywhere — max difference "
                 f"{abs(vfull-vth_l):.1e} V")

    a3 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    Pw = (Vth / (Rth + rl)) ** 2 * rl
    a3.semilogx(rl, Pw * 1e3, color=GREEN, lw=1.6)
    a3.axvline(Rth, color=ORANGE, lw=1.2, ls="--")
    a3.text(Rth * 1.15, Pw.max() * 1e3 * 0.55, "RL = Rth", color=ORANGE, fontsize=7.5)
    a3.plot([RL], [P * 1e3], "o", ms=7, color=DOT)
    a3.set_xlabel("load  RL  (Ω)"); a3.set_ylabel("power in RL  (mW)")
    a3.set_title(f"peak {Vth**2/(4*Rth)*1e3:.4f} mW at RL = Rth, "
                 f"efficiency 50%")

    readout(fig, 0.845, 0.90, [
        "ORIGINAL", "─" * 26,
        f"Vs          {Vs:>10.2f}V",
        f"Ra          {Ra:>10.0f}Ω",
        f"Rb          {Rb:>10.0f}Ω",
        "", "EQUIVALENT", "─" * 26,
        f"Vth (open)  {Vth:>10.5f}V",
        f"Rth (Ra||Rb){Rth:>10.4f}Ω",
        f"Isc         {Vs/Ra*1e3:>10.4f}mA",
        f"Vth/Isc     {Vth/(Vs/Ra):>10.4f}Ω",
        "", "WITH RL", "─" * 26,
        f"RL          {RL:>10.0f}Ω",
        f"v full      {vfull:>10.7f}V",
        f"v Thévenin  {vth_l:>10.7f}V",
        f"difference  {abs(vfull-vth_l):>10.1e}V",
        f"i in RL     {iL*1e3:>10.5f}mA",
        f"P in RL     {P*1e3:>10.5f}mW",
        "", f"P max       {Vth**2/(4*Rth)*1e3:>10.5f}mW",
        f"at RL =     {Rth:>10.2f}Ω",
    ])
    footer(fig, f"Vth = open-circuit voltage  ·  Rth = Vth/Isc = Ra||Rb  ·  "
                f"Pmax = Vth²/4Rth at RL = Rth")
    plt.show()


_p4, _s4 = timeline(399, step=4)
w4 = dict(Vs=widgets.FloatSlider(value=12, min=1, max=24, step=1,
                                 description="source V:", **SL),
          Ra=widgets.FloatSlider(value=470, min=50, max=3000, step=10,
                                 description="Ra (Ω):", **SL),
          Rb=widgets.FloatSlider(value=1000, min=50, max=5000, step=50,
                                 description="Rb (Ω):", **SL),
          RL=widgets.FloatSlider(value=320, min=10, max=5000, step=10,
                                 description="load RL (Ω):", **SL),
          k=_s4)
display(widgets.VBox([widgets.HBox([w4["Vs"], w4["Ra"], w4["Rb"], w4["RL"]]),
                      widgets.HBox([_p4, _s4])]),
        widgets.interactive_output(draw_thevenin, w4))

Output()

## Superposition — one source at a time

In a network of resistors and sources, every branch current is a **linear** function of the sources. So the response to several sources acting together is the sum of the responses to each acting alone:

$$v_{\text{total}}=v\big|_{V_1\text{ only}}+v\big|_{V_2\text{ only}}$$

Killing a source means replacing it by its own zero: a voltage source becomes a **short** (0 V across it), a current source becomes an **open** (0 A through it). Getting that backwards is the classic mistake, and the schematics show which is which.

The three circuits below are solved independently and the readout compares the sum against the joint solution — agreement to $4\times10^{-16}$ V.

Two warnings worth carrying forward. Superposition applies to **voltages and currents, never to power**, because power is quadratic: $(i_1+i_2)^2\neq i_1^2+i_2^2$. And it fails the moment a nonlinear component is present — the diode from the previous notebook breaks it immediately, which is exactly why nonlinear circuits are hard and why mixers work.

In [6]:
def sup_solve(v1, v2, R1, R2, R3):
    G = 1 / R1 + 1 / R2 + 1 / R3
    return (v1 / R1 + v2 / R2) / G


def draw_superposition(k, V1, V2, R1, R2, R3):
    full = sup_solve(V1, V2, R1, R2, R3)
    only1 = sup_solve(V1, 0.0, R1, R2, R3)
    only2 = sup_solve(0.0, V2, R1, R2, R3)
    vmax = max(abs(V1), abs(V2), 1e-9)
    q = k / 400 * 1e-2

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(3, 3, width_ratios=[1.25, 1.1, 0.55],
                          wspace=0.28, hspace=0.5, left=0.02, right=0.995,
                          top=0.90, bottom=0.09)

    scenes = [("both sources", V1, V2, full, True, True),
              (f"only V1 — V2 shorted", V1, 0.0, only1, True, False),
              (f"only V2 — V1 shorted", 0.0, V2, only2, False, True)]
    for r, (nm, a_, b_, vn, on1, on2) in enumerate(scenes):
        a = fig.add_subplot(gs[r, 0])
        sch_axes(a, (-0.4, 5.0), (-0.3, 1.5))
        wire(a, [(0.4, 1.1), (4.4, 1.1)], vn, vmax)
        wire(a, [(0.4, 0.0), (4.4, 0.0)], 0.0, vmax)
        resistor(a, (0.4, 1.1), (1.5, 1.1), a_, vmax, None)
        resistor(a, (3.3, 1.1), (4.4, 1.1), b_, vmax, None)
        resistor(a, (2.4, 1.1), (2.4, 0.0), vn / 2, vmax, None)
        if on1:
            source(a, (0.4, 0.0), (0.4, 1.1), a_ / 2, vmax, "dc", f"{a_:.0f}V")
        else:
            wire(a, [(0.4, 0.0), (0.4, 1.1)], 0.0, vmax)
            a.text(0.4, 1.32, "shorted", color=ORANGE, fontsize=7, ha="center")
        if on2:
            source(a, (4.4, 0.0), (4.4, 1.1), b_ / 2, vmax, "dc", f"{b_:.0f}V")
        else:
            wire(a, [(4.4, 0.0), (4.4, 1.1)], 0.0, vmax)
            a.text(4.4, 1.32, "shorted", color=ORANGE, fontsize=7, ha="center")
        node_dot(a, (2.4, 1.1), vn, vmax)
        charge_dots(a, seg((2.4, 1.1), (2.4, 0.0)), q * vn / R3 * 4e4,
                    spacing=0.26, ms=3.4)
        a.set_title(f"{nm}   →   node = {vn:.6f} V", fontsize=8.2)

    a1 = panel(fig.add_subplot(gs[:, 1]), GREEN)
    a1.bar([0, 1, 2], [only1, only2, only1 + only2],
           color=[POS, PURP, DOT], width=0.6)
    a1.plot([2], [full], "_", ms=42, color=FG, mew=2.5)
    for x, val in ((0, only1), (1, only2), (2, only1 + only2)):
        a1.text(x, val, f"{val:.5f}", ha="center", va="bottom", fontsize=7.5)
    a1.set_xticks([0, 1, 2])
    a1.set_xticklabels(["V1 alone", "V2 alone", "sum"], fontsize=8)
    a1.set_ylabel("node voltage  (V)")
    a1.set_title(f"the white bar is the joint solution\n"
                 f"difference = {abs(full-(only1+only2)):.1e} V")

    p_full = full ** 2 / R3
    p_sum = only1 ** 2 / R3 + only2 ** 2 / R3
    readout(fig, 0.845, 0.90, [
        "SOURCES", "─" * 26,
        f"V1          {V1:>10.2f}V",
        f"V2          {V2:>10.2f}V",
        f"R1 R2 R3    {R1:>4.0f}{R2:>5.0f}{R3:>5.0f}",
        "", "NODE VOLTAGE", "─" * 26,
        f"V1 alone    {only1:>10.7f}V",
        f"V2 alone    {only2:>10.7f}V",
        f"sum         {only1+only2:>10.7f}V",
        f"both at once{full:>10.7f}V",
        f"difference  {abs(full-(only1+only2)):>10.1e}V",
        "", "POWER DOES NOT ADD", "─" * 26,
        f"P joint     {p_full*1e3:>10.5f}mW",
        f"P1 + P2     {p_sum*1e3:>10.5f}mW",
        (f"error       {abs(p_full-p_sum)/p_full*100:>10.2f}%" if p_full > 1e-18
         else "the sources cancel:"),
        ("" if p_full > 1e-18 else "  v = 0 but P1+P2 > 0"),
        "", "killing a source:",
        " voltage src → SHORT",
        " current src → OPEN",
    ], color=ORANGE)
    footer(fig, "superposition holds for v and i, never for power  ·  "
                "and never with a nonlinear element in the circuit")
    plt.show()


_p5, _s5 = timeline(399, step=4)
w5 = dict(V1=widgets.FloatSlider(value=5, min=-10, max=15, step=0.5,
                                 description="V1:", **SL),
          V2=widgets.FloatSlider(value=3, min=-10, max=15, step=0.5,
                                 description="V2:", **SL),
          R1=widgets.FloatSlider(value=1000, min=100, max=5000, step=100,
                                 description="R1 (Ω):", **SL),
          R2=widgets.FloatSlider(value=2000, min=100, max=5000, step=100,
                                 description="R2 (Ω):", **SL),
          R3=widgets.FloatSlider(value=4700, min=100, max=8000, step=100,
                                 description="R3 (Ω):", **SL),
          k=_s5)
display(widgets.VBox([widgets.HBox([w5["V1"], w5["V2"], w5["R1"]]),
                      widgets.HBox([w5["R2"], w5["R3"]]),
                      widgets.HBox([_p5, _s5])]),
        widgets.interactive_output(draw_superposition, w5))

Output()

## Second order — what a circuit does when you hit it with a step

One storage element gives an exponential. **Two** give an oscillation, because energy can now slosh between them, and the whole behaviour collapses onto a single dimensionless number:

$$\zeta=\frac{R}{2}\sqrt{\frac{C}{L}},\qquad
\omega_0=\frac{1}{\sqrt{LC}},\qquad
s_{1,2}=-\zeta\omega_0\pm\omega_0\sqrt{\zeta^2-1}$$

The pole plot is the compact statement. Complex poles ($\zeta<1$) mean ringing at $\omega_d=\omega_0\sqrt{1-\zeta^2}$; as $\zeta$ rises they swing toward the real axis, meet at $\zeta=1$, and split apart along it. The distance from the origin stays $\omega_0$ the whole time — damping rotates the poles, it does not move them nearer or further.

Overshoot follows from $\zeta$ alone, with no reference to $L$, $C$ or $R$ individually:

$$\text{overshoot}=e^{-\pi\zeta/\sqrt{1-\zeta^2}}$$

which gives 72.9% at $\zeta=0.1$, 16.3% at $0.5$, and **4.3% at $\zeta=0.707$** — the flattest response that still gets there quickly, which is why that value turns up as the default in filter design and in the next notebook. Push to $\zeta=1$ and the overshoot vanishes entirely, but arrival is slower, not faster: critical damping is the fastest approach *without* overshoot, not the fastest approach.

In [ ]:
def step_rlc(V, R, L, C, t):
    a = R / (2 * L); w0 = 1 / np.sqrt(L * C)
    if a < w0 - 1e-12:
        wd = np.sqrt(w0 ** 2 - a ** 2)
        return V * (1 - np.exp(-a * t) * (np.cos(wd * t) + a / wd * np.sin(wd * t)))
    if abs(a - w0) < 1e-9 * w0:
        return V * (1 - np.exp(-a * t) * (1 + a * t))
    sr = np.sqrt(a ** 2 - w0 ** 2); s1, s2 = -a + sr, -a - sr
    return V * (1 + (s2 * np.exp(s1 * t) - s1 * np.exp(s2 * t)) / (s1 - s2))


def draw_step2(k, V, R, L_mH, C_uF):
    L, C = L_mH * 1e-3, C_uF * 1e-6
    w0 = 1 / np.sqrt(L * C)
    zeta = (R / 2) * np.sqrt(C / L)
    Rcrit = 2 * np.sqrt(L / C)
    N = 1200
    T = 14 / (zeta * w0) if zeta > 0.08 else 14 / (0.08 * w0)
    t = np.linspace(0, T, N)
    v = step_rlc(V, R, L, C, t)
    kk = int(min(k, N - 1))
    vmax = max(V, 1e-9)
    over = np.exp(-np.pi * zeta / np.sqrt(1 - zeta ** 2)) if zeta < 1 else 0.0

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1.25, 0.55],
                          wspace=0.28, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.5, 4.6), (-0.5, 2.8))
    wire(a0, [(0.5, 2.3), (1.2, 2.3)], V, vmax)
    resistor(a0, (1.2, 2.3), (2.3, 2.3), V, vmax, f"R {R:.1f}Ω")
    inductor(a0, (2.3, 2.3), (3.6, 2.3), v[kk], vmax, f"L {L_mH:.2f}mH")
    wire(a0, [(3.6, 2.3), (4.0, 2.3)], v[kk], vmax)
    wire(a0, [(0.5, 0.2), (4.0, 0.2)], 0.0, vmax)
    source(a0, (0.5, 0.2), (0.5, 2.3), V / 2, vmax, "dc", f"{V:.0f}V step")
    capacitor(a0, (4.0, 2.3), (4.0, 0.2), v[kk] / 2, vmax, f"C {C_uF:.2f}µF")
    node_dot(a0, (4.0, 2.3), v[kk], vmax)
    icur = np.gradient(v, t) * C
    qq = np.cumsum(icur) * (t[1] - t[0])
    charge_dots(a0, loop_rect(0.5, 4.0, 0.2, 2.3),
                qq[kk] / max(abs(qq).max(), 1e-15) * 4.0)
    a0.set_title("a step in, and two storage elements to argue about it")

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(t * 1e6, v, color=POS, lw=1.6)
    a1.axhline(V, color=MUTED, lw=0.9, ls="--")
    if zeta < 1:
        a1.axhline(V * (1 + over), color=ORANGE, lw=0.9, ls=":")
        tp = np.pi / (w0 * np.sqrt(1 - zeta ** 2))
        a1.plot([tp * 1e6], [V * (1 + over)], "o", ms=7, color=ORANGE)
    a1.axvline(t[kk] * 1e6, color=FG, lw=1.0, ls="--")
    a1.set_xlabel("time  (µs)"); a1.set_ylabel("v on C  (V)")
    kind = ("underdamped" if zeta < 0.999 else
            "critically damped" if zeta < 1.001 else "overdamped")
    a1.set_title(f"ζ = {zeta:.4f} — {kind}, overshoot {over*100:.2f}%")

    a2 = panel(fig.add_subplot(gs[1, 1]), PURP)
    th = np.linspace(np.pi / 2, 3 * np.pi / 2, 200)
    a2.plot(w0 * np.cos(th) / 1e3, w0 * np.sin(th) / 1e3, color=GRIDC, lw=1.0)
    if zeta < 1:
        wd = w0 * np.sqrt(1 - zeta ** 2)
        for sgn in (1, -1):
            a2.plot([-zeta * w0 / 1e3], [sgn * wd / 1e3], "x", ms=11, mew=2.4,
                    color=PURP)
    else:
        sr = w0 * np.sqrt(zeta ** 2 - 1)
        for s_ in (-zeta * w0 + sr, -zeta * w0 - sr):
            a2.plot([s_ / 1e3], [0], "x", ms=11, mew=2.4, color=PURP)
    a2.axhline(0, color=GRIDC, lw=0.8); a2.axvline(0, color=GRIDC, lw=0.8)
    a2.set_xlim(-1.6 * w0 / 1e3, 0.3 * w0 / 1e3)
    a2.set_ylim(-1.2 * w0 / 1e3, 1.2 * w0 / 1e3)
    a2.set_xlabel("real  (krad/s)"); a2.set_ylabel("imag  (krad/s)")
    a2.set_title(f"poles sit on a circle of radius ω₀ = {w0/1e3:.2f} krad/s")

    readout(fig, 0.845, 0.90, [
        "CIRCUIT", "─" * 26,
        f"R           {R:>10.2f}Ω",
        f"L           {L_mH:>10.3f}mH",
        f"C           {C_uF:>10.3f}µF",
        "", "DERIVED", "─" * 26,
        f"ω0          {w0/1e3:>10.3f}krad/s",
        f"f0          {w0/2/np.pi/1e3:>10.3f}kHz",
        f"R critical  {Rcrit:>10.3f}Ω",
        f"ζ           {zeta:>10.4f}",
        f"Q = 1/2ζ    {1/(2*max(zeta,1e-9)):>10.3f}",
        "", "STEP RESPONSE", "─" * 26,
        f"overshoot   {over*100:>10.2f}%",
        f"peak value  {V*(1+over):>10.4f}V",
        f"final       {V:>10.4f}V",
        f"v now       {v[kk]:>10.4f}V",
        "", "REFERENCE", "─" * 26,
        "ζ=0.500 → 16.30%",
        "ζ=0.707 →  4.33%",
        "ζ=1.000 →  0.00%",
    ], color=ORANGE if zeta < 1 else FG)
    footer(fig, f"ζ = (R/2)√(C/L) = {zeta:.4f}  ·  ω0 = 1/√(LC) = {w0/1e3:.2f} krad/s  "
                f"·  overshoot = exp(−πζ/√(1−ζ²))")
    plt.show()


_p6, _s6 = timeline(1199, step=12)
w6 = dict(V=widgets.FloatSlider(value=5, min=1, max=12, step=0.5,
                                description="step V:", **SL),
          R=widgets.FloatSlider(value=20, min=0.5, max=300, step=0.5,
                                description="R (Ω):", **SL),
          L_mH=widgets.FloatSlider(value=1.0, min=0.1, max=10, step=0.1,
                                   description="L (mH):", **SL),
          C_uF=widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05,
                                   description="C (µF):", **SL),
          k=_s6)
display(widgets.VBox([widgets.HBox([w6["V"], w6["R"], w6["L_mH"], w6["C_uF"]]),
                      widgets.HBox([_p6, _s6])]),
        widgets.interactive_output(draw_step2, w6))

Output()